# NeuS Training – Google Drive Integration

**Fully automated, unattended training session (~3 hours)**

| Setting | Value |
|---------|-------|
| Dataset | `bmvs_dog` |
| Experiments | `wmask` (with mask) **and** `womask` (without mask) |
| Iterations | 50 000 each |
| Outputs | Saved directly to Google Drive |

### How to use
1. Connect to an **L4 / A100 GPU** runtime for best speed.
2. Click **Runtime → Run all** and leave it running.
3. All checkpoints, logs, meshes and metrics land in `MyDrive/neus_experiments/`.
4. If the session is interrupted, just **Run all** again – training resumes from the last checkpoint.


In [ ]:
# ── Step 1 : Mount Google Drive ───────────────────────────────────────────
from google.colab import drive
import os, time, sys

drive.mount('/content/drive')

# ── Project-level constants (edit DRIVE_DIR if you prefer a different folder)
CASE_NAME = 'bmvs_dog'
DRIVE_DIR = '/content/drive/MyDrive/neus_experiments'
NEUS_DIR  = '/content/NeuS'
DATA_DIR  = f'{NEUS_DIR}/public_data/{CASE_NAME}'

WMASK_EXP_DIR  = f'{DRIVE_DIR}/exp/{CASE_NAME}/wmask'
WOMASK_EXP_DIR = f'{DRIVE_DIR}/exp/{CASE_NAME}/womask'

for d in [
    f'{DRIVE_DIR}/data',
    f'{WMASK_EXP_DIR}/checkpoints',
    f'{WOMASK_EXP_DIR}/checkpoints',
    f'{DRIVE_DIR}/logs',
    f'{DRIVE_DIR}/results/wmask',
    f'{DRIVE_DIR}/results/womask',
]:
    os.makedirs(d, exist_ok=True)

print(f'Drive base  : {DRIVE_DIR}')
print(f'NeuS dir    : {NEUS_DIR}')
print(f'Dataset dir : {DATA_DIR}')
print('✓ Google Drive mounted and project directories ready.')


In [ ]:
# ── Step 2 : Install dependencies ─────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Core NeuS requirements
pip('torch==1.13.1+cu116', 'torchvision==0.14.1+cu116',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu116')
pip('opencv-python==4.7.0.72', 'trimesh==3.9.8', 'PyMCubes==0.1.2',
    'pyhocon==0.3.60', 'icecream', 'tqdm', 'scipy')
# Metrics and visualisation
pip('scikit-image', 'matplotlib')

print('✓ All dependencies installed.')


In [ ]:
# ── Step 3 : Clone NeuS repository ────────────────────────────────────────
import os, subprocess

if not os.path.isdir(NEUS_DIR):
    subprocess.check_call([
        'git', 'clone', 'https://github.com/Totoro97/NeuS.git', NEUS_DIR
    ])
    print('✓ NeuS cloned.')
else:
    print('✓ NeuS directory already present – skipping clone.')

os.chdir(NEUS_DIR)
print(f'Working directory: {os.getcwd()}')


In [ ]:
# ── Step 4 : Download bmvs_dog dataset ─────────────────────────────────────
import os, zipfile, urllib.request, shutil

# Check Drive cache first
DRIVE_DATA_CACHE = f'{DRIVE_DIR}/data/{CASE_NAME}'

if os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join(DATA_DIR, 'image')):
    print(f'✓ Dataset already present at {DATA_DIR}.')
elif os.path.isdir(DRIVE_DATA_CACHE) and os.path.isdir(os.path.join(DRIVE_DATA_CACHE, 'image')):
    # Copy from Drive cache (faster than re-downloading)
    shutil.copytree(DRIVE_DATA_CACHE, DATA_DIR)
    print(f'✓ Dataset copied from Drive cache: {DRIVE_DATA_CACHE}')
else:
    # ── Download from the NeuS public-data Dropbox release ──
    # URL for bmvs_dog (adjust if the Dropbox share URL changes)
    DROPBOX_URL = (
        'https://www.dropbox.com/sh/w0y8bbdmxzik3uk/'
        'AAAaZffBiJevxQzRskoOYcyja/bmvs_dog.zip?dl=1'
    )
    zip_path = f'/tmp/{CASE_NAME}.zip'
    print(f'Downloading {CASE_NAME} from Dropbox...')
    try:
        urllib.request.urlretrieve(DROPBOX_URL, zip_path)
        print('✓ Download complete. Extracting...')
        os.makedirs(f'{NEUS_DIR}/public_data', exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(f'{NEUS_DIR}/public_data')
        os.remove(zip_path)
        # Cache on Drive for future sessions
        shutil.copytree(DATA_DIR, DRIVE_DATA_CACHE)
        print(f'✓ Dataset extracted to {DATA_DIR} and cached on Drive.')
    except Exception as e:
        print(f'⚠ Automatic download failed: {e}')
        print('Please upload the bmvs_dog folder manually to:')
        print(f'  {DATA_DIR}')
        print('or mount a Drive folder that contains it and set DATA_DIR above.')

# Sanity check
if os.path.isdir(os.path.join(DATA_DIR, 'image')):
    n = len(os.listdir(os.path.join(DATA_DIR, 'image')))
    print(f'Dataset OK – {n} training images found.')
else:
    print('⚠ WARNING: Dataset not found. Training will fail.')


In [ ]:
# ── Step 5 : Write experiment configuration files ──────────────────────────
import os

CONF_DIR = f'{NEUS_DIR}/confs_drive'
os.makedirs(CONF_DIR, exist_ok=True)

MODEL_BLOCK = '''
model {
    nerf {
        D = 8,
        d_in = 4,
        d_in_view = 3,
        W = 256,
        multires = 10,
        multires_view = 4,
        output_ch = 4,
        skips=[4],
        use_viewdirs=True
    }

    sdf_network {
        d_out = 257
        d_in = 3
        d_hidden = 256
        n_layers = 8
        skip_in = [4]
        multires = 6
        bias = 0.5
        scale = 1.0
        geometric_init = True
        weight_norm = True
    }

    variance_network {
        init_val = 0.3
    }

    rendering_network {
        d_feature = 256
        mode = idr
        d_in = 9
        d_out = 3
        d_hidden = 256
        n_layers = 4
        weight_norm = True
        multires_view = 4
        squeeze_out = True
    }

    neus_renderer {
        n_samples = 64
        n_importance = 64
        n_outside = 0
        up_sample_steps = 4
        perturb = 1.0
    }
}
'''

# ── wmask config ────────────────────────────────────────────────────────────
wmask_conf = f'''general {{
    base_exp_dir = {WMASK_EXP_DIR}
    recording = [
        ./,
        ./models
    ]
}}

dataset {{
    data_dir = {DATA_DIR}/
    render_cameras_name = cameras_sphere.npz
    object_cameras_name = cameras_sphere.npz
}}

train {{
    learning_rate = 5e-4
    learning_rate_alpha = 0.05
    end_iter = 50000

    batch_size = 512
    validate_resolution_level = 4
    warm_up_end = 5000
    anneal_end = 0
    use_white_bkgd = False

    save_freq = 5000
    val_freq = 2500
    val_mesh_freq = 5000
    report_freq = 100

    igr_weight = 0.1
    mask_weight = 0.1
}}
''' + MODEL_BLOCK

# ── womask config ───────────────────────────────────────────────────────────
womask_conf = f'''general {{
    base_exp_dir = {WOMASK_EXP_DIR}
    recording = [
        ./,
        ./models
    ]
}}

dataset {{
    data_dir = {DATA_DIR}/
    render_cameras_name = cameras_sphere.npz
    object_cameras_name = cameras_sphere.npz
}}

train {{
    learning_rate = 5e-4
    learning_rate_alpha = 0.05
    end_iter = 50000

    batch_size = 512
    validate_resolution_level = 4
    warm_up_end = 5000
    anneal_end = 50000
    use_white_bkgd = False

    save_freq = 5000
    val_freq = 2500
    val_mesh_freq = 5000
    report_freq = 100

    igr_weight = 0.1
    mask_weight = 0.0
}}
''' + MODEL_BLOCK

WMASK_CONF_PATH  = os.path.join(CONF_DIR, 'wmask_drive.conf')
WOMASK_CONF_PATH = os.path.join(CONF_DIR, 'womask_drive.conf')

with open(WMASK_CONF_PATH,  'w') as f: f.write(wmask_conf)
with open(WOMASK_CONF_PATH, 'w') as f: f.write(womask_conf)

# Also copy configs to Drive for reproducibility
import shutil
shutil.copy(WMASK_CONF_PATH,  f'{DRIVE_DIR}/exp/{CASE_NAME}/wmask_drive.conf')
shutil.copy(WOMASK_CONF_PATH, f'{DRIVE_DIR}/exp/{CASE_NAME}/womask_drive.conf')

print('✓ Configuration files written:')
print(f'  wmask  → {WMASK_CONF_PATH}')
print(f'  womask → {WOMASK_CONF_PATH}')
print('✓ Copies saved to Google Drive for reproducibility.')


In [ ]:
# ── Step 6 : Helper utilities ──────────────────────────────────────────────
import re, subprocess, sys, os, time
import numpy as np
import cv2

# Regex for NeuS checkpoint filenames: ckpt_NNNNNN.pth
_CKPT_RE = re.compile(r'^ckpt_(\d+)\.pth$')


def has_checkpoint(exp_dir: str) -> bool:
    """Return True when at least one valid .pth checkpoint exists."""
    ckpt_dir = os.path.join(exp_dir, 'checkpoints')
    if not os.path.isdir(ckpt_dir):
        return False
    return any(_CKPT_RE.match(f) for f in os.listdir(ckpt_dir))


def latest_iter(exp_dir: str) -> int:
    """Return the iteration number of the latest checkpoint (0 if none)."""
    ckpt_dir = os.path.join(exp_dir, 'checkpoints')
    if not os.path.isdir(ckpt_dir):
        return 0
    iters = [
        int(m.group(1))
        for f in os.listdir(ckpt_dir)
        for m in [_CKPT_RE.match(f)] if m
    ]
    return max(iters) if iters else 0


def run_training(conf_path: str, case_name: str, log_path: str,
                 is_continue: bool = False) -> int:
    """
    Run exp_runner.py in a subprocess, streaming stdout/stderr to both
    the notebook cell output and a Drive log file in real time.
    Returns the process exit code.
    """
    cmd = [
        sys.executable, 'exp_runner.py',
        '--mode', 'train',
        '--conf', conf_path,
        '--case', case_name,
    ]
    if is_continue:
        cmd.append('--is_continue')

    print('\n' + '='*70)
    print(f'Command : {" ".join(cmd)}')
    print(f'Log file: {log_path}')
    print('='*70 + '\n')

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'a', buffering=1) as log_f:
        log_f.write(f'\n=== Training started: '
                    f'{time.strftime("%Y-%m-%d %H:%M:%S")} ===\n')
        log_f.write(f'Command: {" ".join(cmd)}\n\n')

        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=NEUS_DIR,
        )

        for line in iter(proc.stdout.readline, ''):
            sys.stdout.write(line)
            sys.stdout.flush()
            log_f.write(line)
            log_f.flush()

        proc.wait()
        status = ('completed successfully' if proc.returncode == 0
                  else f'exited with code {proc.returncode}')
        log_f.write(f'\n=== Training {status}: '
                    f'{time.strftime("%Y-%m-%d %H:%M:%S")} ===\n')

    if proc.returncode == 0:
        print('\n✓ Training completed successfully!')
    else:
        print(f'\n⚠ Training exited with code {proc.returncode}')
    return proc.returncode


print('✓ Helper functions defined.')


In [ ]:
# ── Step 7 : Train WITH mask supervision (wmask) ───────────────────────────
import os

wmask_log  = f'{DRIVE_DIR}/logs/wmask_training.log'
resume     = has_checkpoint(WMASK_EXP_DIR)
iter_start = latest_iter(WMASK_EXP_DIR)

if iter_start >= 50000:
    print(f'✓ wmask already trained to {iter_start} iterations – skipping.')
else:
    if resume:
        print(f'Resuming wmask from iteration {iter_start} ...')
    else:
        print('Starting wmask training from scratch ...')

    t0 = time.time()
    rc = run_training(WMASK_CONF_PATH, CASE_NAME, wmask_log, is_continue=resume)
    elapsed = (time.time() - t0) / 3600
    print(f'wmask wall-clock time: {elapsed:.2f} h')


In [ ]:
# ── Step 8 : Train WITHOUT mask supervision (womask) ───────────────────────
import os

womask_log  = f'{DRIVE_DIR}/logs/womask_training.log'
resume      = has_checkpoint(WOMASK_EXP_DIR)
iter_start  = latest_iter(WOMASK_EXP_DIR)

if iter_start >= 50000:
    print(f'✓ womask already trained to {iter_start} iterations – skipping.')
else:
    if resume:
        print(f'Resuming womask from iteration {iter_start} ...')
    else:
        print('Starting womask training from scratch ...')

    t0 = time.time()
    rc = run_training(WOMASK_CONF_PATH, CASE_NAME, womask_log, is_continue=resume)
    elapsed = (time.time() - t0) / 3600
    print(f'womask wall-clock time: {elapsed:.2f} h')


In [ ]:
# ── Step 9 : Extract final high-resolution meshes ──────────────────────────
import subprocess, sys, os

# Minimum file size (bytes) to consider a mesh valid
_MIN_MESH_BYTES = 1024


def _has_valid_mesh(mesh_dir: str) -> bool:
    """Return True if a non-empty .ply mesh file already exists."""
    if not os.path.isdir(mesh_dir):
        return False
    return any(
        f.endswith('.ply') and os.path.getsize(os.path.join(mesh_dir, f)) > _MIN_MESH_BYTES
        for f in os.listdir(mesh_dir)
    )


def extract_mesh(conf_path: str, case_name: str, exp_dir: str,
                 resolution: int = 512) -> None:
    mesh_dir = os.path.join(exp_dir, 'meshes')
    if _has_valid_mesh(mesh_dir):
        print(f'  Mesh already exists in {mesh_dir} – skipping extraction.')
        return
    cmd = [
        sys.executable, 'exp_runner.py',
        '--mode', 'validate_mesh',
        '--conf', conf_path,
        '--case', case_name,
        '--is_continue',
    ]
    print(f'Extracting mesh (resolution={resolution}) ...')
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=NEUS_DIR)
    if result.returncode == 0 and _has_valid_mesh(mesh_dir):
        plys = [f for f in os.listdir(mesh_dir) if f.endswith('.ply')]
        print(f'  ✓ Mesh saved: {plys}')
    else:
        print(f'  ⚠ Mesh extraction failed or produced an empty file.')
        if result.stderr:
            print(result.stderr[-2000:])


print('── wmask mesh ──')
extract_mesh(WMASK_CONF_PATH, CASE_NAME, WMASK_EXP_DIR)

print('── womask mesh ──')
extract_mesh(WOMASK_CONF_PATH, CASE_NAME, WOMASK_EXP_DIR)

print('✓ Mesh extraction done. Files are on Google Drive.')


In [ ]:
# ── Step 10 : Compute image-quality metrics (SSIM / PSNR / L2) ─────────────
import os
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim_fn

# Maximum PSNR to use when MSE is zero (perfect match)
_PSNR_MAX_DB = 100.0


def compute_metrics(exp_dir: str, n_images: int = 20) -> dict:
    """
    Scans validations_fine/ for the most-recent validation images,
    splits each side-by-side PNG into rendered vs ground-truth halves,
    and returns mean SSIM, PSNR, and L2.
    """
    val_dir = os.path.join(exp_dir, 'validations_fine')
    if not os.path.isdir(val_dir):
        return {}

    # Sort descending so we take the latest n_images
    imgs = sorted(os.listdir(val_dir), reverse=True)[:n_images]
    ssims, psnrs, l2s = [], [], []

    for name in imgs:
        img = cv2.imread(os.path.join(val_dir, name))
        if img is None:
            continue
        w = img.shape[1] // 2
        rendered = img[:, :w].astype(np.float32) / 255.0
        gt       = img[:, w:].astype(np.float32) / 255.0

        # PSNR – cap at _PSNR_MAX_DB to avoid inf skewing the mean
        mse = float(np.mean((rendered - gt) ** 2))
        psnr = _PSNR_MAX_DB if mse == 0 else min(-10.0 * np.log10(mse), _PSNR_MAX_DB)
        psnrs.append(psnr)

        # SSIM – use channel_axis (scikit-image >= 0.19)
        s = ssim_fn(rendered, gt, data_range=1.0, channel_axis=2)
        ssims.append(s)

        # L2
        l2s.append(float(np.sqrt(np.mean((rendered - gt) ** 2))))

    if not psnrs:
        return {}
    return {
        'ssim': float(np.mean(ssims)),
        'psnr': float(np.mean(psnrs)),
        'l2':   float(np.mean(l2s)),
        'n':    len(psnrs),
    }


wmask_metrics  = compute_metrics(WMASK_EXP_DIR)
womask_metrics = compute_metrics(WOMASK_EXP_DIR)

print('\nImage-quality metrics')
print(f'{"":-<50}')
print(f'{"":<10} {"SSIM":>8} {"PSNR":>8} {"L2":>10}')
print(f'{"":-<50}')
for label, m in [("wmask", wmask_metrics), ("womask", womask_metrics)]:
    if m:
        print(f'{label:<10} {m["ssim"]:8.4f} {m["psnr"]:8.2f} {m["l2"]:10.6f}  (n={m["n"]})')
    else:
        print(f'{label:<10}  no validation images found')

# Save metrics JSON to Drive
import json, time
metrics_dict = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'case': CASE_NAME,
    'wmask':  wmask_metrics,
    'womask': womask_metrics,
}
metrics_path = f'{DRIVE_DIR}/results/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f'\n✓ Metrics saved to {metrics_path}')


In [ ]:
# ── Step 11 : Generate difference maps and side-by-side comparisons ─────────
import os
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt


def save_diff_maps(exp_dir: str, out_dir: str,
                   label: str, n_images: int = 5) -> None:
    val_dir = os.path.join(exp_dir, 'validations_fine')
    if not os.path.isdir(val_dir):
        print(f'  No validation images for {label}.')
        return
    os.makedirs(out_dir, exist_ok=True)

    imgs = sorted(os.listdir(val_dir), reverse=True)[:n_images]
    for name in imgs:
        img = cv2.imread(os.path.join(val_dir, name))
        if img is None:
            continue
        w        = img.shape[1] // 2
        rendered = img[:, :w, ::-1].astype(np.float32) / 255.0  # BGR→RGB
        gt       = img[:, w:, ::-1].astype(np.float32) / 255.0

        diff      = np.sqrt(np.sum((rendered - gt) ** 2, axis=2))
        diff_norm = diff / (diff.max() + 1e-8)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(rendered);      axes[0].set_title('Rendered');      axes[0].axis('off')
        axes[1].imshow(gt);            axes[1].set_title('Ground Truth');  axes[1].axis('off')
        im = axes[2].imshow(diff_norm, cmap='hot', vmin=0, vmax=1)
        axes[2].set_title('Difference (L2)'); axes[2].axis('off')
        plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
        plt.suptitle(f'{label} – {name}', fontsize=12)
        plt.tight_layout()
        out_path = os.path.join(out_dir, f'diffmap_{name}')
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.close()
        print(f'  Saved: {out_path}')


print('── wmask difference maps ──')
save_diff_maps(WMASK_EXP_DIR,  f'{DRIVE_DIR}/results/wmask',  'wmask')

print('── womask difference maps ──')
save_diff_maps(WOMASK_EXP_DIR, f'{DRIVE_DIR}/results/womask', 'womask')

# Side-by-side wmask vs womask comparison for latest available image
def _latest_rendered_half(exp_dir):
    """Return the rendered half (RGB float32) of the most recent validation image."""
    val_dir = os.path.join(exp_dir, 'validations_fine')
    if not os.path.isdir(val_dir):
        return None, None
    imgs = sorted(os.listdir(val_dir), reverse=True)
    for name in imgs:
        img = cv2.imread(os.path.join(val_dir, name))
        if img is None:
            continue
        w = img.shape[1] // 2
        rendered = img[:, :w, ::-1].astype(np.float32) / 255.0  # BGR→RGB
        gt_half  = img[:, w:, ::-1].astype(np.float32) / 255.0
        return rendered, gt_half
    return None, None


wm_rendered,  gt_half  = _latest_rendered_half(WMASK_EXP_DIR)
wom_rendered, _        = _latest_rendered_half(WOMASK_EXP_DIR)

if wm_rendered is not None and wom_rendered is not None and gt_half is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(gt_half);       axes[0].set_title('Ground Truth');       axes[0].axis('off')
    axes[1].imshow(wm_rendered);   axes[1].set_title('Rendered (wmask)');   axes[1].axis('off')
    axes[2].imshow(wom_rendered);  axes[2].set_title('Rendered (womask)');  axes[2].axis('off')
    plt.suptitle('wmask vs womask comparison', fontsize=14)
    plt.tight_layout()
    cmp_path = f'{DRIVE_DIR}/results/wmask_vs_womask_comparison.png'
    plt.savefig(cmp_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'✓ Comparison image saved: {cmp_path}')

print('✓ Difference maps generation complete.')


In [ ]:
# ── Step 12 : Generate summary report ─────────────────────────────────────
import os, json, time

# Reload metrics (may have been recomputed in a fresh session)
metrics_path = f'{DRIVE_DIR}/results/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics_dict = json.load(f)
else:
    metrics_dict = {}


def fmt(m, key):
    return f'{m[key]:.4f}' if m and key in m else 'N/A'


wm  = metrics_dict.get('wmask', {})
wom = metrics_dict.get('womask', {})

report_lines = [
    '# NeuS Training Summary Report\n',
    f'Generated : {time.strftime("%Y-%m-%d %H:%M:%S")}\n',
    f'Dataset   : {CASE_NAME}\n',
    f'Drive dir : {DRIVE_DIR}\n',
    '\n',
    '## Training Progress\n',
    f'| Experiment | Iterations completed |\n',
    f'|------------|---------------------|\n',
    f'| wmask      | {latest_iter(WMASK_EXP_DIR):>20,} |\n',
    f'| womask     | {latest_iter(WOMASK_EXP_DIR):>20,} |\n',
    '\n',
    '## Image Quality Metrics\n',
    f'| Experiment | SSIM   | PSNR (dB) | L2      |\n',
    f'|------------|--------|-----------|---------|\n',
    f'| wmask      | {fmt(wm,"ssim"):>6} | {fmt(wm,"psnr"):>9} | {fmt(wm,"l2"):>7} |\n',
    f'| womask     | {fmt(wom,"ssim"):>6} | {fmt(wom,"psnr"):>9} | {fmt(wom,"l2"):>7} |\n',
    '\n',
    '## Output Locations\n',
    f'| Asset | Path |\n',
    f'|-------|------|\n',
    f'| wmask checkpoints | `{WMASK_EXP_DIR}/checkpoints/` |\n',
    f'| womask checkpoints | `{WOMASK_EXP_DIR}/checkpoints/` |\n',
    f'| wmask mesh | `{WMASK_EXP_DIR}/meshes/` |\n',
    f'| womask mesh | `{WOMASK_EXP_DIR}/meshes/` |\n',
    f'| wmask logs | `{DRIVE_DIR}/logs/wmask_training.log` |\n',
    f'| womask logs | `{DRIVE_DIR}/logs/womask_training.log` |\n',
    f'| Difference maps | `{DRIVE_DIR}/results/` |\n',
    f'| Metrics JSON | `{DRIVE_DIR}/results/metrics.json` |\n',
    '\n',
    '## Notes\n',
    '- wmask uses mask supervision (mask_weight=0.1)\n',
    '- womask has no mask supervision (mask_weight=0.0)\n',
    '- Checkpoints saved every 5 000 iterations\n',
    '- To resume after timeout: re-run all cells – '
        'training detects existing checkpoints automatically\n',
]

report_text = ''.join(report_lines)
report_path = f'{DRIVE_DIR}/results/summary_report.md'
with open(report_path, 'w') as f:
    f.write(report_text)

print(report_text)
print(f'✓ Summary report saved to {report_path}')
